# 04 · Análisis de la optimización de hiperparámetros (Workshop S5)
Este cuaderno **no reentrena**: lee los resultados guardados por `03_modelado` y analiza la búsqueda con Optuna.
- Validación interna temporal **C2 → C3**, objetivo **PR-AUC**, 60 trials por modelo (TPE, semilla 42).
- Selección posterior en **C4** con la métrica principal (Lift@k por representante).

Detalle y discusión en [`docs/optimizacion.md`](../docs/optimizacion.md).

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'yaml': 'PyYAML', 'seaborn': 'seaborn'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title Fuente de resultados
FUENTE = "real"  # @param ["real", "sintetica"]
# "real": agregados de LEMAS ya publicados en results/metrics (autorizados; sin datos individuales).
# "sintetica": resultados de ejecutar 03_modelado con la base sintética.
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from src.utils import cargar_config

config = cargar_config()
MET, FIG = RAIZ / config["rutas"]["metricas"], RAIZ / config["rutas"]["figuras"]
DPI = config["figuras"]["dpi"]
sns.set_theme(style="whitegrid")
COLOR, SUAVE = "#8B1E3F", "#9DB4C9"
print("Fuente:", FUENTE, "| métricas en", MET)

## 1. Evolución de la búsqueda

In [ ]:
historial = pd.read_csv(MET / f"optuna_historial_{FUENTE}.csv")
historial["mejor_hasta_ahora"] = historial.groupby("modelo")["value"].cummax()
fig, ax = plt.subplots(figsize=(10, 4))
for modelo, h in historial.groupby("modelo"):
    ax.plot(h["number"], h["mejor_hasta_ahora"], label=modelo)
    ax.scatter(h["number"], h["value"], s=8, alpha=0.3)
ax.set(title="Optimización con Optuna (validación interna C2 → C3)", xlabel="trial",
       ylabel="PR-AUC (puntos: cada trial; línea: mejor acumulado)")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_14_optuna_trials.png", dpi=DPI); plt.show()

resumen = historial.groupby("modelo")["value"].agg(trials="count", minima="min", mediana="median",
                                                   maxima="max").round(4)
resumen["trial_del_maximo"] = historial.loc[historial.groupby("modelo")["value"].idxmax()].set_index("modelo")["number"]
display(resumen)

## 2. Efecto de los hiperparámetros principales
Cada punto es un trial. Permite ver qué zonas del espacio de búsqueda funcionan mejor.

In [ ]:
logistica = historial[historial["modelo"].str.startswith("Regresión")]
arboles = historial[historial["modelo"].str.startswith("Gradient")]
hibrido = historial[historial["modelo"].str.startswith("Logística")]
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(data=logistica, x="params_C", y="value", hue="params_balanceado", ax=ejes[0])
ejes[0].set(xscale="log", title="Regresión logística: C y balanceo", xlabel="C (log)", ylabel="PR-AUC")
sns.scatterplot(data=arboles, x="params_min_samples_leaf", y="value", hue="params_max_depth",
                palette="viridis", ax=ejes[1])
ejes[1].set(title="Gradient boosting: tamaño de hoja y profundidad", xlabel="min_samples_leaf")
sns.scatterplot(data=hibrido, x="params_C", y="value", hue="params_balanceado", ax=ejes[2])
ejes[2].set(xscale="log", title="Logística híbrida: C y balanceo", xlabel="C (log)")
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_15_hiperparametros.png", dpi=DPI); plt.show()

for nombre, datos, col in [("Regresión logística", logistica, "params_conjunto"),
                           ("Regresión logística", logistica, "params_balanceado"),
                           ("Gradient boosting", arboles, "params_conjunto"),
                           ("Logística híbrida", hibrido, "params_balanceado")]:
    print(f"{nombre} · mediana de PR-AUC por {col.replace('params_', '')}:",
          datos.groupby(col)["value"].median().round(4).to_dict())

## 3. De la validación interna a C4
La optimización elige con C3; la decisión final se toma en C4, con datos que no participaron en la búsqueda.

In [ ]:
seleccion = pd.read_csv(MET / f"seleccion_C4_{FUENTE}.csv")
ajuste = pd.read_csv(MET / f"diagnostico_ajuste_{FUENTE}.csv")
display(seleccion)
display(ajuste)

fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
orden = seleccion.sort_values("lift_k")
ejes[0].barh(orden["candidato"], orden["lift_k"],
             color=[COLOR if t != "regla" else SUAVE for t in orden["tipo"]])
ejes[0].axvline(1, ls="--", c="gray"); ejes[0].set(title="Lift@k en C4", xlabel="Lift@k (1 = azar)")
x = np.arange(len(ajuste))
ejes[1].bar(x - 0.2, ajuste["pr_auc_entrenamiento"], 0.4, label="entrenamiento", color=SUAVE)
ejes[1].bar(x + 0.2, ajuste["pr_auc_C4"], 0.4, label="C4", color=COLOR)
ejes[1].set_xticks(x, [m.split(" (")[0] for m in ajuste["modelo"]], rotation=10)
ejes[1].set(title="Diagnóstico de ajuste (semana 3)", ylabel="PR-AUC"); ejes[1].legend()
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_16_ajuste_c4.png", dpi=DPI); plt.show()

## 4. Conclusiones
1. La búsqueda mejoró la PR-AUC interna (en datos reales, la regresión logística pasó de 0,062 a 0,106), pero **el orden de los modelos cambió de C3 a C4**: con pocos eventos y tasas que varían entre años, lo óptimo en una cohorte no se traslada a la siguiente.
2. **Gradient boosting se sobreajusta** (brecha de PR-AUC 0,14 entre entrenamiento y C4) y **los modelos lineales se quedan cortos** (subajuste por falta de señal).
3. Por eso la selección se hizo en C4 frente a reglas sin IA, y la regla **D2** resultó la mejor opción, confirmada después en C5 (cuaderno 05).